## AI Insights — Claude-Powered Healthcare Analysis

In [0]:
import anthropic
import os
from datetime import datetime
from pyspark.sql import Row

client = anthropic.Anthropic(
    api_key=dbutils.secrets.get(scope="healthflow", key="anthropic-key")
)

# Load gold tables
conditions = spark.table("healthflow_catalog.gold.conditions_analysis").toPandas().to_string()
hospitals  = spark.table("healthflow_catalog.gold.hospital_performance").toPandas().to_string()

# Generate insights with Claude
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""You are a healthcare data analyst.
Analyze this hospital data and be concise:

CONDITIONS DATA:
{conditions}

HOSPITAL DATA:
{hospitals}

Provide:
1. Top 3 critical insights
2. Billing anomalies
3. Patient risk patterns
4. Actionable recommendations
""",
    }],
)

report = response.content[0].text
print(report)

# Persist the report as a Delta table row
report_df = spark.createDataFrame([
    Row(report_text=report, generated_at=datetime.now().isoformat())
])
report_df.write.format("delta").mode("overwrite").saveAsTable(
    "healthflow_catalog.gold.ai_insights"
)

print("\n✓ AI insights saved to healthflow_catalog.gold.ai_insights")

# Healthcare Data Analysis: Critical Insights

## 🔴 TOP 3 CRITICAL INSIGHTS

### 1. Extreme Billing Variability Signals Systemic Issues
Average billing across conditions is remarkably uniform (~$25,100-$25,800), yet **individual hospital bills range from -$2,008 to $52,374**. Negative billing values exist (e.g., Clark-Espinoza: -$2,008, Group Smith: -$1,016), indicating **billing system errors or improper credits** requiring immediate audit.

### 2. Urgent Care Concentration Risk
Several hospitals carry disproportionately high urgent case loads:
- **Group Davis**: 10 urgent/20 patients (50% urgent rate)
- **Inc Smith**: 15 urgent/33 patients (45%)
- **LLC Johnson**: 11 urgent/30 patients (37%)
- **Group Johnson**: 13 urgent/27 patients (48%)

These facilities are **understaffed relative to acuity demand**.

### 3. Condition Prevalence Essentially Equal — Screening Gap
All 6 conditions show nearly identical patient counts (9,095–9,218), which is **statistically improbable** in a natural

Trace(trace_id=tr-1d98e9d524c1d16adc4b8f7bb8191379)